In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path


load_dotenv(Path("../.env")) 

# === Config MySQL ===
host     = os.getenv("MYSQL_HOST")
user     = os.getenv("MYSQL_USER")
pwd = os.getenv("MYSQL_PASSWORD")
db       = os.getenv("MYSQL_DB")
port=3306

In [1]:

import duckdb
import pymysql

# Étapes S3 pour DuckDB
duckdb.sql("INSTALL httpfs;")
duckdb.sql("LOAD httpfs;")
duckdb.sql("SET s3_region='us-east-1';")
duckdb.sql("SET s3_url_style='path';")

# Lecture des données depuis S3 (colonnes filtrées)
query = """
    SELECT
        fsq_place_id,
        name,
        latitude,
        longitude,
        address,
        fsq_category_ids,
        locality,
        region
    FROM read_parquet(
        's3://fsq-os-places-us-east-1/release/dt=2025-05-09/places/parquet/*.parquet',
        union_by_name=true
    )
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL AND country= 'FR' AND locality IS NOT NULL AND region IS NOT NULL AND address IS NOT NULL
"""
results_fsq = duckdb.sql(query).fetchall()


In [ ]:
import duckdb
import pymysql

# Étapes S3 pour DuckDB
duckdb.sql("INSTALL httpfs;")
duckdb.sql("LOAD httpfs;")
duckdb.sql("SET s3_region='us-east-1';")
duckdb.sql("SET s3_url_style='path';")

# Lecture des données depuis S3 (colonnes filtrées)
query = """
    SELECT
        *
    FROM read_parquet(
        's3://fsq-os-places-us-east-1/release/dt=2025-09-09/places/parquet/*.parquet',
        union_by_name=true
    )
"""
results_fsq = duckdb.sql(query).fetchall()

In [ ]:
def connexion_database():
    try:
            
        conx = pymysql.connect(
            host=host, 
            user=user, 
            password=pwd, 
            database=database, 
            port=port
        )

        print(" Connexion réussie !")

        # Test simple
        with conx.cursor() as cursor:
            cursor.execute("SELECT VERSION()")
            version = cursor.fetchone()
            print(f"Version MySQL : {version[0]}")

        return conx
    except pymysql.Error as e:
        print(f"❌ Erreur : {e}")


In [ ]:
#Creation des insert
try:
    conx = connexion_database()
    cursor= conx.cursor()
    
    lst_insert=list()
    match_cities = {}
    partial_match_cities={}
    #for row in results_fsq:
        
    # Assignations poi
    """idFsq = row[0]
    name = row[1]
    lat = row[2]
    long= row[3]
    address=row[4]
    idCity=""
    """  
        
        
    # Trouver la ville
    #slct_city = "SELECT idCity, name FROM ville WHERE name LIKE %s AND departement LIKE %s"
    slct_cities = "SELECT idCity, name, region FROM ville"
    nb_resp = cursor.execute(slct_cities)
    all_cities = cursor.fetchall()
    
    #nb_resp = cursor.execute(slct_city, (row[6].replace(" ", '%'), row[7]))
    for idCity, name, department in all_cities:
        
        # Match total
        key = (name.lower().strip(), departement.lower().strip())
        match_cities[key]=idCity
        
        
        # Match partiel
        key_p = (city_name.lower().replace(" ", ""), departement.lower().strip())
        if key_p not in partial_match_cities:
            partial_match_cities[key_p]=[]
        partial_match_cities.append(idCity)
    print(f"Cache créé avec {len(match_cities)} villes")
    

    
    
        
                
    # Si on a une ville, on Insert le poi à la liste
    """if idCity != "":
        print("insertion", address, lat, long )
        lst_insert.append((idFsq, name, idCity, address, lat, long))
    """
        #idPoi = cursor.lastrowid
            
        # Insert les catégories
    """cats = row[5]
        for cat in cats:
            insrt_cat = "INSERT INTO poi_categorie(idCat, idPoi) VALUES(%s, %s)"
            cursor.execute(insrt_cat, (cat, idPoi))    """


    # Insertions des poi en base
    print("lancment des insertions multiples")
            
    """    
    insrt_poi = "INSERT INTO poi(idFsq, name, idCity, address, latitudePoi, longitudePoi) VALUES(%s, %s, %s, %s, %s, %s)"
    cursor.executemany(insrt_poi, lst_insert)   
    print("insertion complete")
    conx.commit()
    print("commit complete")
    conx.close()"""
except Exception as e:
        print(e)

 Connexion réussie !
Version MySQL : 8.0.41
name 'departement' is not defined
